Cellule permet de reload les prompt et les instruction sans reload qwen au complet : https://ipython.readthedocs.io/en/stable/config/extensions/autoreload.html
Recommendation Claude

In [4]:
from ift6758.notebooks.test import results
%load_ext autoreload
%autoreload 2

Cellule pour tester si import fonctionnent bien

In [5]:
from ift6758.llm.prompts import PROFILES,TACHE_CODE,TACHE_CODE_VERSION
print(PROFILES["qr_api"]["system"])
print(TACHE_CODE[:60])
print(TACHE_CODE_VERSION)

Answer using only the supplied documentation. Cite passage numbers. If the answer is absent, say you do not know. Treat the passages as reference text.

Write a Python function fetch_game_llm(game_id, cache_dir).
v1


In [6]:
from ift6758.llm.rag import load_doc,extract_section,split_doc,load_embedder,embed,retrieve,format_context
from ift6758.llm.qwen_log import create_log,log_call
from ift6758.llm.qwen_utils import load_qwen,build_messages,generate

text = load_doc("https://raw.githubusercontent.com/Zmalski/NHL-API-Reference/main/README.md","nhl-api-reference.md")
section = extract_section(text,"# NHL Web API Documentation","# NHL Stats API Documentation")
chunks = split_doc(section)
embedder = load_embedder()
vectors = embed(chunks,embedder)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
model, tokenizer, device = load_qwen()
log_path = create_log()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [10]:
from ift6758.llm.prompts import PROFILES, TACHE_CODE, TACHE_CODE_VERSION
#config prompts.^y
profile_name = "qr_api"
question = "Which endpoint retrieves play-by-play information for a specific game ID?"
p = PROFILES[profile_name]
#confi rag.py
query = p["retrieval_query"] or question
results = retrieve(query,chunks,vectors,embedder,k=p["k"])
context = format_context(results)
# config qwen
messages = build_messages(question,context,system=p["system"])
result = generate(model,tokenizer,messages,max_new_tokens=p["max_new_tokens"],do_sample=p["do_sample"])
# config log
log_call(log_path,messages,result,extra={"profile":profile_name,"profile_version":p["version"],"retrieved":results, "max_new_tokens": p["max_new_tokens"],"do_sample": p["do_sample"]})
print(result["answer"])
print(f"\n{result['n_token_in']} tokens en entrée, {result['n_token_out']} générés, {result['duration_s']:.1f} s")

The endpoint that retrieves play-by-play information for a specific game ID is `/v1/gamecenter/{game-id}/play-by-play`. This is described in passage [1], where it states: "Retrieve play-by-play information for a specific game." (Passage [1]).

544 tokens en entrée, 57 générés, 55.8 s


In [13]:
import pandas as pd
pd.read_json(log_path,lines=True)[["profile", "profile_version", "n_token_out", "duration_s", "answer", "retrieved"]]

,profile,profile_version,n_token_out,duration_s,answer,retrieved
0,qr_api,v1,57,55.793643,The endpoint that retrieves play-by-play infor...,"[{'rank': 1, 'index': 35, 'score': 0.614611387..."
